<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%202/1_st445_week2_lab1_api_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUTOR COPY: SOLUTIONS AND NOTES

**Timing (about 50 minutes):** Part A 20 min, Part B 20 min, Part C 10 min in the seminar (run C1 to C3 together; C4 to C6 after).

**Why Our World in Data in Part C:** this lab uses the WHO GHO API with World Bank denominators. Part C teaches the pipeline (collect, prepare, join on `iso3` and `year`, validate, weight by the right denominator) with the OWID chart API. OWID also introduces a CSV response and non-ISO `OWID_` codes.

**Running Part A:** paste every printed URL into a browser.

**Before the seminar:** run the notebook once with a connection so `cache/` is populated. Check A2: at the time of writing, `/country/France/...` gives status 200 with error 120 in the body. In C1, check that the CSV's value column for `co-emissions-per-capita` is still a single column, and in C3 which OWID-only codes appear today.

**Where most get stuck:**
* A2 and B1: a country name instead of a code gives error 120.
* B2: thinking `per_page` changes the total.
* B4: not passing `page`, so the loop repeats page 1 (the 98-record test catches it).
* B5: confusing `record_path` with `meta`.
* C1: keeping continents (no code) or `OWID_WRL`; or dropping Kosovo by filtering before recoding. Order matters: recode, then filter.
* C2: World Bank aggregates (`WLD`, `HIC`) pass a "three capital letters" test; only the `/country` metadata removes them. The C5 test catches them.
* C3: computing the match rate over all outer-join rows. The rate that matters is the share of outcome rows matched.
* C4 check 5: the unweighted high-income mean is inflated by small, very high emitters. Ask what each average answers.
* C5 step 6: the per-person figures differ slightly because OWID divides by UN population estimates, not World Bank ones. Large differences for small territories are expected; the point is to notice and explain, not to force agreement.

# ST445 Week 2 seminar, lab 1: collecting data from APIs

**Seminar lab · about 50 minutes, with Part C to finish afterwards · no Python experience beyond the pre-class activities needed**

In this lab you build API requests **one piece at a time** and watch how each piece changes what comes back. You then learn the routine professionals follow for every request, turn the answers into tables, and finish with research tasks that use the same skills your project will need.

The main source is the **World Bank API**, which publishes economic and social indicators for every country. You also use **Open-Meteo** (weather) and, in Part C, the **Our World in Data chart API**, which you join to World Bank data to practice joining two sources. None of the APIs needs a key.

### How the lab is organised

| Part | What happens | Time |
|---|---|---|
| **A. Guided demo** | Your tutor runs these cells with you. Run each one when they do, and paste the printed URLs into your browser. | 20 min |
| **B. Pause and analyse** | Short exercises: change one thing, predict the result, complete small pieces of code. | 20 min |
| **C. Two APIs, one dataset** | Collect from the Our World in Data and World Bank APIs, join on country and year, and validate the result. | 10 min in the seminar, then after |

Cell marks: 🟢 run and read · 🟡 fill the gaps marked `____` · 🔵 write the code yourself · ✍️ write a short answer in the Markdown cell. Test cells print ✅ or ❌ with a hint. Boxes titled **Method card** explain each new function: what it does and the minimum you need to set.

**Before you start:** **File → Save a copy in Drive**, and rename it `<yourname>_week2_api_lab.ipynb`.

## Set-up

`requests` sends web requests; `pandas` makes tables; the rest are small tools from Python's standard library. The two helper functions at the bottom save you typing later: `check` prints ✅ or ❌, and `filled` stops a cell politely if you have not replaced a `____` yet.

In [1]:
# 🟢 Run this first
import json, time, hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
import pandas as pd

HEADERS = {"User-Agent": "ST445 seminar lab (London School of Economics)"}

def check(condition, hint="Not yet. Read the task again."):
    print("✅ Correct" if condition else f"❌ {hint}")

def filled(*values):
    """False (with a reminder) if any value still contains ____."""
    if any("____" in str(v) for v in values):
        print("✋ Replace every ____ first, then run this cell again.")
        return False
    return True

print("Ready.")

Ready.


---
# Part A. Guided demo

## A1. Building a request, one piece at a time

Every API request is a web address made of three pieces:

| Piece | Meaning | World Bank example |
|---|---|---|
| **Base URL** | where the API lives; the same for every request | `https://api.worldbank.org/v2` |
| **Endpoint** | *what* you are asking for; a path added to the base | `/country/GBR/indicator/SP.POP.TOTL` |
| **Parameters** | *how* you want it; `?` then `name=value` pairs joined by `&` | `?format=json&per_page=5` |

In the endpoint, `GBR` is the United Kingdom's three-letter ISO 3166 code (a Week 1 standard), and `SP.POP.TOTL` is the World Bank's code for total population.

### Step 1: base URL plus endpoint

Run the cell, then **copy the printed URL into a new browser tab** and look at what comes back.

In [2]:
# 🟢 Step 1
BASE = "https://api.worldbank.org/v2"
endpoint = "/country/GBR/indicator/SP.POP.TOTL"

url = BASE + endpoint
print(url)

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL


**What you should see in the browser:** a page of **XML**, a tag-based format like HTML. The World Bank sends XML unless you ask for something else. The numbers are there, but XML is awkward to work with in Python. So we add our first parameter.

### Step 2: add a parameter, `format=json`

Parameters start with `?`. Paste the new URL into your browser.

In [3]:
# 🟢 Step 2
url = BASE + endpoint + "?format=json"
print(url)

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json


**What you should see:** the same data as **JSON**. It starts with `[{"page":1,"pages":2,"per_page":50,"total":65,...}` and continues with a list of records, one per year, newest first. (Your numbers may differ slightly as new years are published.) By default you get **50 records per page**.

### Step 3: a second parameter, `per_page=5`

Extra parameters are joined with `&`. Predict what will change before you paste it.

In [4]:
# 🟢 Step 3
url = BASE + endpoint + "?format=json&per_page=5"
print(url)

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=5


**What you should see:** only **5 records**, and the first part now says `"per_page":5` and many more `"pages"`. Same data, cut into smaller pages.

### Step 4: a third parameter, `date=2015:2020`

The documentation says `date` takes a single year or a range written `start:end`.

In [5]:
# 🟢 Step 4
url = BASE + endpoint + "?format=json&per_page=5&date=2015:2020"
print(url)

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=5&date=2015:2020


**What you should see:** 6 years (2015 to 2020), so `"total":6` and, with 5 per page, `"pages":2`. Each parameter changed one thing about the answer.

### Step 5: let `requests` build the URL for you

Typing `?` and `&` by hand is error-prone. In practice you put the parameters in a **dictionary** and let `requests` build the URL. The cell below builds exactly the same URL as Step 4, without sending anything yet, so you can compare.

In [6]:
# 🟢 Step 5
params = {"format": "json", "per_page": 5, "date": "2015:2020"}
prepared = requests.Request("GET", BASE + endpoint, params=params).prepare()
print(prepared.url)
print("Same as Step 4:", prepared.url == BASE + endpoint + "?format=json&per_page=5&date=2015:2020")

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=5&date=2015%3A2020
Same as Step 4: False


From now on, every cell prints the URL it is about to request, so you can always paste it into a browser and see the raw answer for yourself. Here is a small helper that does exactly that.

In [7]:
# 🟢 A helper you will use all lab
def show_url(base, endpoint, params=None):
    """Print and return the full URL that requests would send."""
    full = requests.Request("GET", base + endpoint, params=params).prepare().url
    print(full)
    return full

show_url(BASE, "/country/FRA/indicator/SP.POP.TOTL", {"format": "json", "per_page": 3})

https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&per_page=3


'https://api.worldbank.org/v2/country/FRA/indicator/SP.POP.TOTL?format=json&per_page=3'

## A2. The professional routine for every request

Getting *a* response is easy. Getting a response you can **trust** takes the same five steps every time. Professionals do them in this order, because each step protects the next:

| Step | What you do | Why |
|---|---|---|
| 1 | **Build and print the URL** | You can check it by eye and paste it into a browser |
| 2 | **Send the request with a timeout** | A slow server cannot freeze your notebook |
| 3 | **Check the status code before anything else** | 200 means the server handled it; 4xx means your request is wrong; 5xx means the server has a problem |
| 4 | **Check the body has the shape the documentation promises** | Some APIs report errors inside a 200 response (you will see one in a moment) |
| 5 | **Only then use the data, and save the raw response** | You never analyse an error message by mistake, and you can always go back to what you received |

> **Method card: `requests.get(url, params=None, headers=None, timeout=None)`**
>
> * `url` (**required**): base plus endpoint, without the `?...` part.
> * `params`: a dictionary of parameters; `requests` adds `?` and `&`.
> * `headers`: a dictionary of extra information; we send a `User-Agent` saying who we are.
> * `timeout`: seconds to wait before giving up. **Always set it**; without it, a request can hang for ever.
>
> It returns a **response** object `r`. The parts you need: `r.status_code` (a number such as 200), `r.url` (the URL actually sent), `r.headers["Content-Type"]` (what kind of data came back), and `r.json()` (the body turned into Python lists and dictionaries; only call it once you know the body is JSON).

Here are the five steps written out, one line each, for population in the UK:

In [8]:
# 🟢 The five steps, one at a time
params = {"format": "json", "per_page": 5, "date": "2015:2020"}

# 1. Build and print the URL
show_url(BASE, endpoint, params)

# 2. Send it, with a timeout
r = requests.get(BASE + endpoint, params=params, headers=HEADERS, timeout=30)

# 3. Check the status code first
print("Status code:", r.status_code)
if r.status_code != 200:
    raise RuntimeError(f"Request failed with status {r.status_code}")

# 4. Check the body is the shape the documentation promises: a list of [metadata, records]
body = r.json()
print("Content type:", r.headers.get("Content-Type"))
print("Body is a list of length", len(body))

# 5. Only now use the data
metadata, records = body[0], body[1]
print("Metadata:", metadata)
print("First record:", records[0])

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=5&date=2015%3A2020
Status code: 200
Content type: application/json;charset=utf-8
Body is a list of length 2
Metadata: {'page': 1, 'pages': 2, 'per_page': 5, 'total': 6, 'sourceid': '2', 'lastupdated': '2026-10-08'}
First record: {'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'}, 'country': {'id': 'GB', 'value': 'United Kingdom'}, 'countryiso3code': 'GBR', 'date': '2020', 'value': 66740000, 'unit': '', 'obs_status': '', 'decimal': 0}


**What you should see:** status **200**; content type `application/json`; a list of length **2**. The first element is the **metadata** (page numbers and totals); the second is the list of **records**. A record looks like:

```
{"indicator": {"id": "SP.POP.TOTL", "value": "Population, total"},
 "country": {"id": "GB", "value": "United Kingdom"},
 "countryiso3code": "GBR", "date": "2020", "value": 67081234, ...}
```

Two Week 1 ideas are visible: `date` is **text**, not a number; and `indicator` and `country` are small dictionaries **nested** inside each record.

### When step 4 matters: an error hidden in a 200

Now ask for a country that does not exist, writing the name instead of the code. Predict the status code first.

In [9]:
# 🟢 Predict, then run
bad_params = {"format": "json"}
show_url(BASE, "/country/France/indicator/SP.POP.TOTL", bad_params)
r = requests.get(BASE + "/country/France/indicator/SP.POP.TOTL", params=bad_params,
                 headers=HEADERS, timeout=30)
print("Status code:", r.status_code)
print("Body:", r.text[:300])

https://api.worldbank.org/v2/country/France/indicator/SP.POP.TOTL?format=json
Status code: 200
Body: [{"message":[{"id":"120","key":"Invalid value","value":"The provided parameter value is not valid"}]}]


**What you should see:** status **200**, yet the body is an error message:

```
[{"message": [{"id": "120", "key": "Invalid value", "value": "The provided parameter value is not valid"}]}]
```

**Error 120** means one of the values in your URL is not valid: here, `France` is not a country code. (The World Bank accepts `FRA` or `FR`.) The server *handled* the request, so it says 200, but you did not get data. That is why step 4 exists. If you meet error 120 later in this lab, check the spelling of every code in your URL.

### Putting the routine into one function

Rather than repeat five steps in every cell, professionals wrap them in a function. The function below also saves each response in a `cache/` folder and uses the saved copy if the network fails, which keeps a busy seminar room working.

In [10]:
# 🟢 The routine as a function
CACHE = Path("cache"); CACHE.mkdir(exist_ok=True)

def get_worldbank(endpoint, params, name):
    """Steps 1 to 5 for the World Bank API. Returns (metadata, records)."""
    show_url(BASE, endpoint, params)                                        # 1
    path = CACHE / f"{name}.json"
    try:
        r = requests.get(BASE + endpoint, params=params, headers=HEADERS, timeout=30)   # 2
        if r.status_code != 200:                                            # 3
            raise RuntimeError(f"Status {r.status_code}: {r.text[:200]}")
        body = r.json()
        path.write_bytes(r.content)                                         # 5: keep the raw response
    except requests.RequestException as err:
        if not path.exists():
            raise
        print(f"(Network problem: {type(err).__name__}. Using the saved copy.)")
        body = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(body, list) and body and "message" in body[0]:            # 4
        msg = body[0]["message"][0]
        raise ValueError(f"World Bank error {msg['id']}: {msg['value']}. Check every code in the URL.")
    return body[0], body[1] or []

metadata, records = get_worldbank(endpoint, {"format": "json", "per_page": 50}, "gbr_population_p1")
print(f"Page {metadata['page']} of {metadata['pages']}, {len(records)} records here, {metadata['total']} in total")

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=50
Page 1 of 2, 50 records here, 66 in total


## A3. Getting every page

With 50 records per page and 65 years of data, the answer comes in **2 pages**. `metadata["pages"]` tells you how many, and the `page` parameter asks for a particular one. The loop below asks for page 1, then page 2, and so on, until it has them all, pausing half a second between requests to be polite.

In [11]:
# 🟢 Follow every page
all_records = []
page = 1
while True:
    metadata, records = get_worldbank(endpoint, {"format": "json", "per_page": 50, "page": page},
                                      f"gbr_population_p{page}")
    all_records.extend(records)                 # add this page's records to the list
    if page >= metadata["pages"]:               # stop after the last page
        break
    page += 1
    time.sleep(0.5)

print("Collected", len(all_records), "records; the API says there are", metadata["total"])
check(len(all_records) == metadata["total"], "The loop should collect every record.")

https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=50&page=1
https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&per_page=50&page=2
Collected 66 records; the API says there are 66
✅ Correct


**Professional habit:** compare what you collected with the total the API reports. It is a one-line test that the loop did not stop early.

## A4. From JSON records to a table

`all_records` is a list of dictionaries, and some values are themselves dictionaries. First try the obvious approach:

In [12]:
# 🟢 The obvious approach
plain = pd.DataFrame(all_records)
plain[["indicator", "country", "date", "value"]].head(3)

,indicator,country,date,value
0,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'GB', 'value': 'United Kingdom'}",2025,69487000
1,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'GB', 'value': 'United Kingdom'}",2024,69281000
2,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'GB', 'value': 'United Kingdom'}",2023,68526000


**What you should see:** the `indicator` and `country` columns contain whole **dictionaries** in each cell, such as `{'id': 'GB', 'value': 'United Kingdom'}`. You cannot filter or group on those easily. `pd.json_normalize` flattens them into ordinary columns.

> **Method card: `pd.json_normalize(data, record_path=None, meta=None, sep=".", max_level=None)`**
>
> * `data` (**required**): a list of dictionaries (or one dictionary). For a simple list of records, **this is all you need to set.**
> * `sep`: the character joining nested key names. The default `"."` gives columns like `country.value`; setting `sep="_"` gives `country_value`, which is easier to type. **Set this almost always.**
> * `record_path`: use only when each record contains a *list* you want to turn into rows (example in B5). It names the key holding that list.
> * `meta`: use with `record_path`, to keep fields from the outer record next to each inner row.
> * `max_level`: how many levels deep to flatten. Leave it out to flatten everything.
>
> Minimum for today: `pd.json_normalize(records, sep="_")`.

Before flattening, write down what one row will be: **one row is the value of one indicator, for one country, in one year.**

In [13]:
# 🟢 Flatten, then choose and tidy the columns
pop = pd.json_normalize(all_records, sep="_")
print(pop.columns.tolist())

pop = pop[["country_value", "countryiso3code", "indicator_id", "date", "value"]]
pop["date"] = pop["date"].astype(int)             # text to whole numbers
pop["population_m"] = pop["value"] / 1e6           # millions, easier to read
print(pop.dtypes)
pop.sort_values("date").tail()

['countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal', 'indicator_id', 'indicator_value', 'country_id', 'country_value']
country_value       object
countryiso3code     object
indicator_id        object
date                 int64
value                int64
population_m       float64
dtype: object


,country_value,countryiso3code,indicator_id,date,value,population_m
4,United Kingdom,GBR,SP.POP.TOTL,2021,66978000,66.978
3,United Kingdom,GBR,SP.POP.TOTL,2022,67636000,67.636
2,United Kingdom,GBR,SP.POP.TOTL,2023,68526000,68.526
1,United Kingdom,GBR,SP.POP.TOTL,2024,69281000,69.281
0,United Kingdom,GBR,SP.POP.TOTL,2025,69487000,69.487


**What you should see:** column names such as `indicator_id`, `indicator_value`, `country_id` and `country_value`; after conversion, `date` is an integer and `value` a float. UK population in recent years is about 68 to 69 million. If the latest year has no figure yet, its `value` shows `NaN`.

> **Method card: `series.astype(type)`** converts a column to another type, for example `astype(int)` for whole numbers or `astype(float)`. It fails loudly if a value cannot be converted, which is what you want: better an error than a silent mistake.

## A5. Keep the raw data and a provenance record

Your project must show where each dataset came from, when, and under what licence. Save the records exactly as received, and add one line to a provenance file. World Bank Open Data is published under the **Creative Commons Attribution 4.0** licence (CC BY 4.0): you may reuse it, with attribution.

In [14]:
# 🟢 Save raw data and a provenance line
RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")      # ISO 8601, in UTC
raw_file = RAW / f"worldbank_GBR_SP.POP.TOTL_{stamp}.json"
raw_file.write_text(json.dumps(all_records), encoding="utf-8")

provenance = {"collected_at": stamp, "source": "World Bank API v2",
              "url": BASE + endpoint, "params": {"format": "json", "per_page": 50},
              "records": len(all_records), "file": raw_file.name,
              "sha256": hashlib.sha256(raw_file.read_bytes()).hexdigest(),
              "licence": "CC BY 4.0 (World Bank Open Data)"}
with open("data/provenance.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(provenance) + "\n")
print(json.dumps(provenance, indent=2))

{
  "collected_at": "2026-10-13T004031Z",
  "source": "World Bank API v2",
  "url": "https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL",
  "params": {
    "format": "json",
    "per_page": 50
  },
  "records": 66,
  "file": "worldbank_GBR_SP.POP.TOTL_2026-10-13T004031Z.json",
  "sha256": "b2cccaf6e565fd7601e00e327622e2365ac3400a0f25a55c1027197cdb8f9f66",
  "licence": "CC BY 4.0 (World Bank Open Data)"
}


That is the guided demo: building a URL piece by piece, the five-step routine, pages, flattening, and provenance. Part B asks you to do each of these yourself.

---
# Part B. Pause and analyse

Work with the person next to you. In every exercise, **paste the printed URL into your browser** to see the raw answer as well.

## B1. Change one piece at a time

The World Bank's code for annual **inflation** (consumer prices, %) is `FP.CPI.TOTL.ZG`. France's three-letter country code is `FRA`.

🟡 Fill the gaps to get France's inflation, **10 records per page**. Then answer the question below the cell.

In [15]:
country = "FRA"           # a three-letter code, such as GBR, not a country name
indicator = "FP.CPI.TOTL.ZG"
per_page = 10

if filled(country, indicator, per_page):
    metadata, records = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                      {"format": "json", "per_page": per_page}, f"{country}_{indicator}_p1")
    fra = pd.json_normalize(records, sep="_")
    print(fra[["country_value", "date", "value"]])
    check(set(fra["countryiso3code"]) == {"FRA"} and set(fra["indicator_id"]) == {"FP.CPI.TOTL.ZG"}
          and len(fra) == 10, "FRA, FP.CPI.TOTL.ZG and 10 per page.")

https://api.worldbank.org/v2/country/FRA/indicator/FP.CPI.TOTL.ZG?format=json&per_page=10
  country_value  date     value
0        France  2025  0.943825
1        France  2024  1.997426
2        France  2023  4.880002
3        France  2022  5.222941
4        France  2021  1.640906
5        France  2020  0.477701
6        France  2019  1.106766
7        France  2018  1.852221
8        France  2017  1.029206
9        France  2016  0.184450
✅ Correct


If you see **World Bank error 120**, a value in the URL is not valid: check that the country is a code (`FRA`) and not a name, and that the indicator code has no typos. Look at the printed URL in your browser to see the error for yourself.

✍️ **Analyse.** With `per_page` set to 10, how many pages does France's inflation series have (`metadata["pages"]`)? Which piece of the URL would you change to get only 2010 to 2019?

*Your answer:*

## B2. Predict the answer from the URL

For each URL, predict `total` and `pages` **before** you run the cell, then check by running it (or by pasting the URL into your browser).

| URL parameters | Your predicted `total` | Your predicted `pages` |
|---|---|---|
| `format=json&per_page=10&date=2010:2019` | | |
| `format=json&per_page=4&date=2010:2019` | | |
| `format=json&per_page=4&date=2019` | | |

In [16]:
# 🟢 Run after predicting
for p in [{"format": "json", "per_page": 10, "date": "2010:2019"},
          {"format": "json", "per_page": 4, "date": "2010:2019"},
          {"format": "json", "per_page": 4, "date": "2019"}]:
    meta, recs = get_worldbank("/country/FRA/indicator/FP.CPI.TOTL.ZG", p,
                               f"fra_cpi_{p['per_page']}_{p['date'].replace(':', '-')}")
    print(f"   total={meta['total']}, pages={meta['pages']}\n")

https://api.worldbank.org/v2/country/FRA/indicator/FP.CPI.TOTL.ZG?format=json&per_page=10&date=2010%3A2019
   total=10, pages=1

https://api.worldbank.org/v2/country/FRA/indicator/FP.CPI.TOTL.ZG?format=json&per_page=4&date=2010%3A2019
   total=10, pages=3

https://api.worldbank.org/v2/country/FRA/indicator/FP.CPI.TOTL.ZG?format=json&per_page=4&date=2019
   total=1, pages=1



**Key output:** 10 records in 1 page; 10 records in 3 pages (4 + 4 + 2); 1 record in 1 page. `date` decides **which** records exist; `per_page` only decides how they are **split**.

## B3. Two APIs, two ways of reporting an error

✍️ Before running the cell, predict: what status code will **Open-Meteo** return for a latitude of 999, which cannot exist? Write it here: ____

In [17]:
# 🟢 Run after predicting
OM = "https://api.open-meteo.com/v1"
p = {"latitude": 999, "longitude": 0, "current": "temperature_2m"}
show_url(OM, "/forecast", p)
r = requests.get(OM + "/forecast", params=p, headers=HEADERS, timeout=30)
print("Status code:", r.status_code)
print("Body:", r.text[:200])

https://api.open-meteo.com/v1/forecast?latitude=999&longitude=0&current=temperature_2m
Status code: 400
Body: {"reason":"Latitude must be in range of -90 to 90°. Given: 999.0.","error":true}


**Key output:** Open-Meteo returns **400** with a JSON explanation that the latitude is out of range. That is the usual design: the status code alone tells you something is wrong (step 3). The World Bank's 200-with-a-message (A2) is easier to miss, which is why step 4 exists.

✍️ **Reflect (one sentence).** Why is it a professional habit to check the status code *before* calling `r.json()`?

*Your answer:*

## B4. Complete the page loop as a function

🟡 Turn the A3 loop into a function you can reuse for any country and indicator. Three gaps: the `page` parameter, the stopping test, and what to return.

In [18]:
def fetch_indicator(country, indicator, date=None, per_page=50):
    """All records for a country (or several, joined with ';') and an indicator, following every page."""
    params = {"format": "json", "per_page": per_page}
    if date is not None:
        params["date"] = date
    collected, page = [], 1
    while True:
        meta, recs = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                   {**params, "page": page}, f"{country}_{indicator}_{date}_{per_page}_p{page}")
        collected.extend(recs)
        if page >= meta["pages"]:
            break
        page += 1
        time.sleep(0.5)
    return collected

Several countries can go in one request, separated by `;`. The test asks for **GDP growth** (`NY.GDP.MKTP.KD.ZG`) for the seven G7 countries over 14 years, with only 20 records per page, so the function has to follow 5 pages.

In [19]:
# Test for B4
G7 = "CAN;FRA;DEU;ITA;JPN;GBR;USA"
growth = fetch_indicator(G7, "NY.GDP.MKTP.KD.ZG", date="2010:2023", per_page=20)
growth_df = pd.json_normalize(growth, sep="_")
check(len(growth_df) == 98, f"Expected 7 countries x 14 years = 98 records, got {len(growth_df)}. Is the page number passed to the request?")

https://api.worldbank.org/v2/country/CAN;FRA;DEU;ITA;JPN;GBR;USA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=20&date=2010%3A2023&page=1
https://api.worldbank.org/v2/country/CAN;FRA;DEU;ITA;JPN;GBR;USA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=20&date=2010%3A2023&page=2
https://api.worldbank.org/v2/country/CAN;FRA;DEU;ITA;JPN;GBR;USA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=20&date=2010%3A2023&page=3
https://api.worldbank.org/v2/country/CAN;FRA;DEU;ITA;JPN;GBR;USA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=20&date=2010%3A2023&page=4
https://api.worldbank.org/v2/country/CAN;FRA;DEU;ITA;JPN;GBR;USA/indicator/NY.GDP.MKTP.KD.ZG?format=json&per_page=20&date=2010%3A2023&page=5
✅ Correct


## B5. Practise `json_normalize`'s parameters

Some APIs put a **list inside each record**. Here is a small, invented example: each country record holds a list of yearly readings.

In [20]:
# 🟢 A small nested example
nested = [
    {"code": "GBR", "name": "United Kingdom", "readings": [{"year": 2022, "value": 9.1}, {"year": 2023, "value": 7.3}]},
    {"code": "FRA", "name": "France",         "readings": [{"year": 2022, "value": 5.2}, {"year": 2023, "value": 4.9}]},
]
print(pd.json_normalize(nested, sep="_"))

  code            name                                           readings
0  GBR  United Kingdom  [{'year': 2022, 'value': 9.1}, {'year': 2023, ...
1  FRA          France  [{'year': 2022, 'value': 5.2}, {'year': 2023, ...


**What you should see:** 2 rows, and a `readings` column holding a whole list in each cell. `json_normalize` flattens nested *dictionaries* on its own, but a nested *list* needs `record_path`.

🟡 Use `record_path` to get **one row per reading**, and `meta` to keep each country's `code` and `name` next to its readings.

In [21]:
long = pd.json_normalize(nested, record_path="readings", meta=["code", "name"])
print(long)
check(len(long) == 4 and set(long.columns) == {"year", "value", "code", "name"},
      "record_path names the key holding the list; meta lists the outer fields to keep.")

   year  value code            name
0  2022    9.1  GBR  United Kingdom
1  2023    7.3  GBR  United Kingdom
2  2022    5.2  FRA          France
3  2023    4.9  FRA          France
✅ Correct


✍️ **Analyse.** In `pd.json_normalize(nested, sep="_")`, one row is one country. In `long`, what is one row? Which one would you use to compute each country's average, and which to count how many readings there are?

*Your answer:*

## B6. Long or wide?

`growth_df` from B4 is **long**: one row per country–year. Comparing countries side by side is easier **wide**: one row per year, one column per country.

> **Method card: `df.pivot(index, columns, values)`**
>
> * `index` (**required**): the column whose values become the **rows** (here `date`).
> * `columns` (**required**): the column whose values become the **column headings** (here `countryiso3code`).
> * `values` (**required**): the column that fills the table (here `value`).
>
> It fails if two rows share the same index and column, which is a useful check that one row really was one country–year.

🟡 Fill the gaps, then find the country with the lowest growth in 2020.

In [22]:
growth_df["date"] = growth_df["date"].astype(int)
wide = growth_df.pivot(index="date", columns="countryiso3code", values="value")
print(wide.round(1).tail())

lowest_2020 = wide.loc[2020].idxmin()     # idxmin: the column with the smallest value in that row
print("Lowest growth in 2020:", lowest_2020)
check(wide.shape == (14, 7), "index='date', columns='countryiso3code', values='value'.")

countryiso3code  CAN  DEU  FRA   GBR  ITA  JPN  USA
date                                               
2019             1.9  1.0  2.0   1.3  0.4 -0.3  2.6
2020            -5.0 -4.1 -7.4 -10.0 -8.9 -4.3 -2.1
2021             6.0  3.9  6.9   8.5  8.9  3.6  6.2
2022             4.7  1.8  2.7   5.1  4.8  1.3  2.5
2023             2.0 -0.9  1.4   0.3  0.9  0.7  2.9
Lowest growth in 2020: GBR
✅ Correct


## B7. A new API: build its URL piece by piece

Open-Meteo has a **historical weather** API. Its documentation is at https://open-meteo.com/en/docs/historical-weather-api. Use it to find the parameter names, then build the request in steps, printing the URL each time and pasting it into your browser.

| Piece | Value |
|---|---|
| Base URL | `https://archive-api.open-meteo.com/v1` |
| Endpoint | `/archive` |
| Parameters you need | `latitude`, `longitude`, `start_date`, `end_date`, `daily`, `timezone` |

🟡 Complete the parameters for **London** (latitude 51.5144, longitude −0.1165), **1 to 31 July 2025**, with the **daily maximum temperature** (`temperature_2m_max`) and times in `Europe/London`.

In [23]:
ARCHIVE = "https://archive-api.open-meteo.com/v1"
p = {"latitude": 51.5144, "longitude": -0.1165,
     "start_date": "2025-07-01", "end_date": "2025-07-31",
     "daily": "temperature_2m_max", "timezone": "Europe/London"}

if filled(*p.values()):
    show_url(ARCHIVE, "/archive", p)
    r = requests.get(ARCHIVE + "/archive", params=p, headers=HEADERS, timeout=30)
    print("Status code:", r.status_code)                 # step 3 first
    if r.status_code == 200:
        july = r.json()
        print(list(july.keys()))                          # step 4: what shape came back?
        temps = pd.DataFrame({"date": pd.to_datetime(july["daily"]["time"]),
                              "tmax_c": july["daily"]["temperature_2m_max"]})
        print(temps.describe())
        check(len(temps) == 31, "start_date 2025-07-01, end_date 2025-07-31, daily temperature_2m_max.")

https://archive-api.open-meteo.com/v1/archive?latitude=51.5144&longitude=-0.1165&start_date=2025-07-01&end_date=2025-07-31&daily=temperature_2m_max&timezone=Europe%2FLondon
Status code: 200
['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']
                      date     tmax_c
count                   31  31.000000
mean   2025-07-16 00:00:00  24.396774
min    2025-07-01 00:00:00  19.600000
25%    2025-07-08 12:00:00  21.950000
50%    2025-07-16 00:00:00  23.600000
75%    2025-07-23 12:00:00  25.600000
max    2025-07-31 00:00:00  33.600000
std                    NaN   3.396811
✅ Correct


**Key output:** status 200, keys including `daily` and `daily_units`, and a table of **31 days**. Open-Meteo returns **parallel lists** (`"time": [...]` and `"temperature_2m_max": [...]`) rather than a list of records, so `pd.DataFrame` with one column per list is the right tool, and `json_normalize` is not needed. Printing the keys (step 4) is how you find out.

✍️ **Experiment.** Change `daily` to `temperature_2m_max,precipitation_sum` (two variables, separated by a comma). Paste the new URL into your browser. What changes in the answer?

*Your answer:*

---
# Part C. Two APIs, one validated dataset

Real projects rarely use one source. A typical pattern: an **outcome** indicator from one organisation, plus **context and denominators** (population, GDP, income group) from another, joined on **country (ISO3 code) and year**. Your project follows a similar pattern.

The two sources:

* the **Our World in Data (OWID) chart API**, which serves the data behind every chart on ourworldindata.org, here carbon dioxide (CO₂) emissions from the Global Carbon Budget;
* the **World Bank Indicators API**, as in Parts A and B.

| Stage | What you do | Section |
|---|---|---|
| Collect | One indicator from OWID; matching series from the World Bank | C1, C2 |
| Prepare | Make both tables share the same key columns, with the same types, and one row per key | C1, C2 |
| Join | Merge on `iso3` and `year`, and look at what did **not** match | C3 |
| Validate | Check keys, match rate, ranges, missing values, a spot value, and the right denominator | C4 |

The worked example (C1 to C4) asks: **how do CO₂ emissions per person differ across World Bank income groups, 2015 to 2019?** Then you repeat the pipeline yourself (C5) and turn the checks into a reusable function (C6).

## C1. Worked example: one indicator from the Our World in Data API

### Building the request

Every OWID chart has a page, and the API is that page's address with an ending added:

| Piece | Value | Meaning |
|---|---|---|
| Base URL | `https://ourworldindata.org/grapher` | the chart API |
| Endpoint | `/co-emissions-per-capita.csv` | one chart (its **slug**), as **CSV** data |
| or | `/co-emissions-per-capita.metadata.json` | the same chart's **metadata**: title, units, source, citation |
| Parameters | `v=1`, `csvType=full`, `useColumnShortNames=true` | API version; every country and year (not just what the chart shows); short column names |

To find a slug, search https://ourworldindata.org/data, open a chart, and copy the last part of its address. The helper below builds both URLs and prints them for your browser.

In [24]:
# 🟢 Build OWID URLs
OWID = "https://ourworldindata.org/grapher"
OWID_PARAMS = {"v": 1, "csvType": "full", "useColumnShortNames": "true"}

def owid_urls(slug):
    """Print and return the data (CSV) and metadata (JSON) URLs for one OWID chart."""
    data_url = show_url(OWID, f"/{slug}.csv", OWID_PARAMS)
    meta_url = show_url(OWID, f"/{slug}.metadata.json", OWID_PARAMS)
    return data_url, meta_url

owid_urls("co-emissions-per-capita")

https://ourworldindata.org/grapher/co-emissions-per-capita.csv?v=1&csvType=full&useColumnShortNames=true
https://ourworldindata.org/grapher/co-emissions-per-capita.metadata.json?v=1&csvType=full&useColumnShortNames=true


('https://ourworldindata.org/grapher/co-emissions-per-capita.csv?v=1&csvType=full&useColumnShortNames=true',
 'https://ourworldindata.org/grapher/co-emissions-per-capita.metadata.json?v=1&csvType=full&useColumnShortNames=true')

Open both URLs in your browser. **What you should see:** the first downloads (or displays) a **CSV file**, a format you have not met yet in this lab: plain text, one row per line, values separated by commas. Its first line names the columns: `Entity` (a country or a group), `Code`, `Year`, and one value column with a short name, holding tonnes of CO₂ per person. The next lines look like `Afghanistan,AFG,2019,0.3`. The second URL returns JSON describing the chart: its title, units, and the **citation** you must give.

### Collecting with the professional routine

Same five steps as before. The only difference is step 4: the body is CSV, so we check the content type and read it with `pd.read_csv`.

> **Method card: `pd.read_csv(file)`** reads comma-separated text into a table. To read text you already have in memory (such as `r.text`), wrap it in `StringIO(...)`, which makes text behave like a file.

In [25]:
# 🟢 The routine for the OWID API
from io import StringIO

def get_owid(slug):
    """Data and metadata for one OWID chart, with status checks and a saved copy of each."""
    data_url, meta_url = owid_urls(slug)
    out = {}
    for kind, url, ext in [("data", data_url, "csv"), ("meta", meta_url, "json")]:
        cache_file = CACHE / f"owid_{slug}.{ext}"
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            print(f"{kind}: status {r.status_code}, content type {r.headers.get('Content-Type')}")   # steps 3 and 4
            if r.status_code != 200:
                raise RuntimeError(f"Status {r.status_code}: {r.text[:200]}")
            cache_file.write_bytes(r.content)                          # keep the raw response
            text = r.text
        except requests.RequestException as err:
            if not cache_file.exists():
                raise
            print(f"(Network problem: {type(err).__name__}. Using the saved copy.)")
            text = cache_file.read_text(encoding="utf-8")
        out[kind] = pd.read_csv(StringIO(text)) if ext == "csv" else json.loads(text)
    return out["data"], out["meta"]

co2_raw, co2_meta = get_owid("co-emissions-per-capita")
print("\nTitle:   ", co2_meta["chart"]["title"])
print("Citation:", co2_meta["chart"].get("citation"))
co2_raw.head()

https://ourworldindata.org/grapher/co-emissions-per-capita.csv?v=1&csvType=full&useColumnShortNames=true
https://ourworldindata.org/grapher/co-emissions-per-capita.metadata.json?v=1&csvType=full&useColumnShortNames=true
data: status 200, content type text/csv
meta: status 200, content type application/json

Title:    CO₂ emissions per capita
Citation: Global Carbon Budget (2025); Population based on various sources (2024)


,entity,code,year,emissions_total_per_capita
0,Afghanistan,AFG,1949,0.001992
1,Afghanistan,AFG,1950,0.010837
2,Afghanistan,AFG,1951,0.011625
3,Afghanistan,AFG,1952,0.011468
4,Afghanistan,AFG,1953,0.013123


**What you should see:** status 200 twice, content types `text/csv` and `application/json`, the chart's title and citation, and the first rows of the table. Keep the citation: it goes into the provenance record and your report.

### Preparing the OWID table

The table mixes **countries** with **groups**. Look at what has no ordinary three-letter code:

In [26]:
# 🟢 Look first: entities without an ordinary ISO3 code
co2_raw.columns = co2_raw.columns.str.lower()             # Entity, Code, Year -> entity, code, year
odd = co2_raw[~co2_raw["code"].fillna("").str.fullmatch(r"[A-Z]{3}")]
print(odd.drop_duplicates("entity")[["entity", "code"]].head(15))

                              entity       code
76                            Africa   OWID_AFR
1057                            Asia   OWID_ASI
1287    Asia (excl. China and India)        NaN
7685                          Europe   OWID_EUR
7915            Europe (excl. EU-27)        NaN
8145            Europe (excl. EU-28)        NaN
8375             European Union (27)  OWID_EU27
8600             European Union (28)        NaN
10884          High-income countries   OWID_HIC
13088                         Kosovo   OWID_KOS
14131           Low-income countries   OWID_LIC
14247  Lower-middle-income countries   OWID_LMC
17214                  North America   OWID_NAM
17440      North America (excl. USA)        NaN
18141                        Oceania   OWID_OCE


**What you should see:** two kinds of non-country rows.

* Groups with **no code at all**: continents such as *Africa* and *Europe*, and groupings such as *High-income countries* or *European Union (27)*.
* Codes beginning **`OWID_`**: OWID's own codes for entities that have no ISO code, such as `OWID_WRL` (World). One of them **is** a country: `OWID_KOS` is Kosovo, which the World Bank codes `XKX`.

So the preparation has four decisions, each written down:

1. **Recode** the one country with a non-ISO code (`OWID_KOS` to `XKX`). The recode table goes into the provenance record.
2. **Keep countries only:** rows whose code is three capital letters.
3. **Rename and keep** the key columns and the value: `iso3`, `year` (integer), `co2_per_capita`, and keep only 2015 to 2019.
4. **One row is one country–year.** Check it.

In [27]:
# 🟢 Prepare the OWID table
OWID_TO_WB = {"OWID_KOS": "XKX"}                                             # 1. recode
value_col = [c for c in co2_raw.columns if c not in ("entity", "code", "year")]
check(len(value_col) == 1, "Expected exactly one value column.")

co2 = co2_raw.assign(code=co2_raw["code"].replace(OWID_TO_WB))
co2 = co2[co2["code"].fillna("").str.fullmatch(r"[A-Z]{3}")]                 # 2. countries only
co2 = co2.rename(columns={"code": "iso3", value_col[0]: "co2_per_capita"})   # 3. rename
co2 = co2[["iso3", "year", "co2_per_capita"]]
co2["year"] = co2["year"].astype(int)
YEARS = range(2015, 2020)
co2 = co2[co2["year"].isin(YEARS)]

check(not co2.duplicated(["iso3", "year"]).any(), "One row should be one country-year.")    # 4.
print(co2["iso3"].nunique(), "countries,", len(co2), "rows")

✅ Correct
✅ Correct
214 countries, 1070 rows


**What you should see:** about 220 countries and territories over 5 years, and both checks ✅.

> **Method card: `df.duplicated(columns)`** is True for each row whose values in those columns already appeared above it. If a key is duplicated, a join multiplies rows silently, so this check comes **before** every join.

## C2. Worked example: context and the right denominator from the World Bank

The outcome is CO₂ **per person**, so the right denominator for weighting is the **population** (`SP.POP.TOTL`). We also take **GDP per capita** (`NY.GDP.PCAP.CD`) as context, and each country's **income group**.

`wb_series` is the page loop from A3 as a function, with 1,000 records per page to keep the number of requests small.

In [28]:
# 🟢 World Bank series for all countries
def wb_series(indicator, date, country="all"):
    """Every record for one World Bank indicator, following every page."""
    collected, page = [], 1
    while True:
        meta, recs = get_worldbank(f"/country/{country}/indicator/{indicator}",
                                   {"format": "json", "per_page": 1000, "date": date, "page": page},
                                   f"wb_{country}_{indicator}_{date.replace(':', '-')}_p{page}")
        collected.extend(recs)
        if page >= meta["pages"]:
            return collected
        page += 1
        time.sleep(5)

def wb_table(records, value_name):
    """Records to a table with the shared key names and types."""
    t = pd.json_normalize(records, sep="_")
    t = t.rename(columns={"countryiso3code": "iso3", "date": "year", "value": value_name})
    t["year"] = t["year"].astype(int)                  # the World Bank sends years as text
    return t[["iso3", "year", value_name]]

date = "2015:2019"
pop = wb_table(wb_series("SP.POP.TOTL", date), "population")
gdppc = wb_table(wb_series("NY.GDP.PCAP.CD", date), "gdp_per_capita_usd")
print(len(pop), "population rows;", len(gdppc), "GDP per capita rows")

https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=1000&date=2015%3A2019&page=1
https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=1000&date=2015%3A2019&page=2
https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=1000&date=2015%3A2019&page=1
https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=1000&date=2015%3A2019&page=2
1330 population rows; 1330 GDP per capita rows


The World Bank has the same trap as OWID: `country/all` returns **aggregates** (`WLD`, `EUU`, `HIC`...). They have three-letter codes, so the "three capital letters" test does not catch them. Its `/country` endpoint marks them with the region `Aggregates`, and gives each country's income group.

In [29]:
# 🟢 Countries only, with income group, one row per country-year
meta, country_list = get_worldbank("/country", {"format": "json", "per_page": 400}, "wb_countries")
countries = pd.json_normalize(country_list, sep="_")
countries = countries[countries["region_value"] != "Aggregates"]
countries = countries.rename(columns={"id": "iso3", "name": "country", "incomeLevel_value": "income_group"})
countries = countries[["iso3", "country", "income_group"]]

wb = pop.merge(gdppc, on=["iso3", "year"], how="outer")
wb = wb[wb["iso3"].isin(countries["iso3"])].merge(countries, on="iso3", how="left")
check(not wb.duplicated(["iso3", "year"]).any(), "One row should be one country-year.")
print(wb["iso3"].nunique(), "countries,", len(wb), "rows")
wb.head()

https://api.worldbank.org/v2/country?format=json&per_page=400
✅ Correct
217 countries, 1085 rows


,iso3,year,population,gdp_per_capita_usd,country,income_group
0,ABW,2015,107906.0,27458.220154,Aruba,High income
1,ABW,2016,108727.0,27441.550214,Aruba,High income
2,ABW,2017,108735.0,28440.041688,Aruba,High income
3,ABW,2018,108908.0,30082.158423,Aruba,High income
4,ABW,2019,109203.0,30654.485124,Aruba,High income


**What you should see:** about 217 countries over 5 years. Both tables now have `iso3` (text) and `year` (integer) with exactly these names. The types matter: `2019` and `"2019"` are **not** equal, so a join on mismatched types finds nothing.

## C3. The join

> **Method card: `pd.merge(left, right, on, how, validate, indicator)`**
>
> * `left`, `right` (**required**): the two tables. Put the **outcome** table on the left.
> * `on` (**required here**): the key columns, with the same names in both: `["iso3", "year"]`.
> * `how`: `"inner"` keeps rows in both; `"left"` keeps every row of `left`; `"outer"` keeps every row of either.
> * `validate="one_to_one"`: raise an error if either table has a duplicated key. **Always set it.**
> * `indicator=True`: adds a `_merge` column, `both`, `left_only` or `right_only`. **Set it while you check a join.**

Start with an **outer** join so you can see what failed to match. The match rate that matters is the share of **outcome rows** (the OWID table) that found a World Bank partner.

In [30]:
# 🟢 Join, keeping everything, and look at the result
check_join = pd.merge(co2, wb, on=["iso3", "year"], how="outer", validate="one_to_one", indicator=True)
print(check_join["_merge"].value_counts(), "\n")

print("In OWID but not the World Bank:", sorted(check_join.loc[check_join["_merge"] == "left_only", "iso3"].unique()))
print("In the World Bank but not OWID:", sorted(check_join.loc[check_join["_merge"] == "right_only", "iso3"].unique()))
match_rate = (check_join["_merge"] == "both").sum() / len(co2)
print(f"\nShare of OWID rows matched: {match_rate:.1%}")

_merge
both          1025
right_only      60
left_only       45
Name: count, dtype: int64 

In OWID but not the World Bank: ['AIA', 'BES', 'COK', 'MSR', 'NIU', 'SHN', 'SPM', 'TWN', 'WLF']
In the World Bank but not OWID: ['ASM', 'CHI', 'CYM', 'GIB', 'GUM', 'IMN', 'MAF', 'MCO', 'MNP', 'PRI', 'SMR', 'VIR']

Share of OWID rows matched: 95.8%


**What you should see:** most rows `both`, a high match rate, and two short lists. Read them; they are information, not errors:

* **OWID only:** small territories that the World Bank does not list separately, such as `AIA` (Anguilla), `COK` (Cook Islands) or `NIU` (Niue).
* **World Bank only:** places for which the Global Carbon Budget gives no separate figure in these years.

Kosovo is **not** in either list, because we recoded `OWID_KOS` in C1. For the analysis we keep the **inner** join.

In [31]:
# 🟢 The analysis table
merged = check_join[check_join["_merge"] == "both"].drop(columns="_merge")
print(merged.shape)
merged.head()

(1025, 7)


,iso3,year,co2_per_capita,population,gdp_per_capita_usd,country,income_group
0,ABW,2015,8.614971,107906.0,27458.220154,Aruba,High income
1,ABW,2016,8.410633,108727.0,27441.550214,Aruba,High income
2,ABW,2017,8.420440,108735.0,28440.041688,Aruba,High income
3,ABW,2018,8.157750,108908.0,30082.158423,Aruba,High income
4,ABW,2019,7.766699,109203.0,30654.485124,Aruba,High income


## C4. Validate the joined data

A join that runs is not a join that is right. Five checks, each a few lines. They are what your project's reproducibility evidence should contain.

In [32]:
# 🟢 Checks 1 to 3: keys and match rate; plausible ranges; missing values
check(not merged.duplicated(["iso3", "year"]).any(), "No duplicated keys after the join.")
check(match_rate > 0.9, "Fewer OWID rows matched than expected: check key types, names and codes.")

check(merged["co2_per_capita"].between(0, 60).all(), "CO2 per person outside 0-60 tonnes: a unit or code problem?")
check((merged["population"].dropna() > 0).all(), "Population must be positive.")

print("\nMissing values per column:\n", merged.isna().sum())
print("\nRows per year:\n", merged.groupby("year").size())

✅ Correct
✅ Correct
✅ Correct
✅ Correct

Missing values per column:
 iso3                   0
year                   0
co2_per_capita         0
population             0
gdp_per_capita_usd    20
country                0
income_group           0
dtype: int64

Rows per year:
 year
2015    205
2016    205
2017    205
2018    205
2019    205
dtype: int64


**What you should see:** all checks ✅, a few missing GDP values (not every country reports every year), and roughly the same number of rows each year. A year with far fewer rows would mean one source has not published it yet.

In [33]:
# 🟢 Check 4: a spot check against both sources, by hand
print(merged[(merged["iso3"] == "GBR") & (merged["year"] == 2019)].T, "\n")
print("OWID chart (hover over the United Kingdom, 2019): https://ourworldindata.org/grapher/co-emissions-per-capita?country=GBR")
show_url(BASE, "/country/GBR/indicator/SP.POP.TOTL", {"format": "json", "date": "2019"})

                               359
iso3                           GBR
year                          2019
co2_per_capita            5.435078
population              66628000.0
gdp_per_capita_usd    43160.684397
country             United Kingdom
income_group           High income 

OWID chart (hover over the United Kingdom, 2019): https://ourworldindata.org/grapher/co-emissions-per-capita?country=GBR
https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&date=2019


'https://api.worldbank.org/v2/country/GBR/indicator/SP.POP.TOTL?format=json&date=2019'

Open both in a browser and compare the values with the table. If they match, the whole chain (request, reading the CSV, recode, filter, rename, type conversion, join) worked for at least one known case. Record the spot check in your report.

### Check 5: the right denominator

The average of countries' per-person emissions treats Qatar and China alike. Weighting each country by its **population** gives the emissions of the average **person** in the income group, which is the same as the group's total emissions divided by its total population. Because the outcome is *per person*, population is the right denominator.

In [34]:
# 🟢 Check 5: unweighted versus population-weighted, by income group (2019)
y = merged[merged["year"] == 2019].dropna(subset=["population"])
summary = y.groupby("income_group").apply(
    lambda g: pd.Series({"countries": len(g),
                         "mean_of_countries": g["co2_per_capita"].mean(),
                         "population_weighted": (g["co2_per_capita"] * g["population"]).sum() / g["population"].sum()}),
    include_groups=False).round(2)
summary

,countries,mean_of_countries,population_weighted
income_group,,,
High income,74.0,9.29,10.63
Low income,25.0,0.34,0.33
Lower middle income,47.0,1.09,1.43
Upper middle income,59.0,3.92,5.54


**What you should see:** emissions per person rising steeply with income group, and the two averages differing. In the high-income group, small oil and gas producers with very high emissions per person pull the unweighted mean up; weighting by population brings the value closer to the large economies. A plausible gradient across income groups is also a final sanity check on the join: if low-income countries came out highest, something upstream would be wrong.

### Save with provenance

Save the joined table, and one provenance line per source with its licence, citation and the recode table. Both OWID and the World Bank publish under **CC BY 4.0**; OWID also asks you to cite the **underlying source** (here the Global Carbon Budget), which is the citation in the metadata.

In [35]:
# 🟢 Save
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
Path("data/processed").mkdir(parents=True, exist_ok=True)
merged.to_csv("data/processed/co2_population_gdp.csv", index=False)
with open("data/provenance.jsonl", "a", encoding="utf-8") as f:
    for src in [{"source": "Our World in Data chart API", "chart": "co-emissions-per-capita", "years": "2015-2019",
                 "records": len(co2), "licence": "CC BY 4.0", "citation": co2_meta["chart"].get("citation")},
                {"source": "World Bank API v2", "indicator": "SP.POP.TOTL; NY.GDP.PCAP.CD", "years": date,
                 "records": len(pop) + len(gdppc), "licence": "CC BY 4.0"}]:
        f.write(json.dumps({"collected_at": stamp, **src, "join": "inner on iso3, year",
                            "recode": OWID_TO_WB, "match_rate_outcome_rows": round(match_rate, 3)}) + "\n")
print("Saved", len(merged), "rows")

Saved 1025 rows


## C5. Your turn: total emissions, and a cross-source check

Repeat the pipeline with a different OWID chart, then use the World Bank denominator to **check one source against the other**.

**Question:** what share of the world's CO₂ emissions came from each World Bank income group in 2020 to 2022, and does dividing OWID's total emissions by World Bank population reproduce OWID's own per-person figures?

1. 🔵 **Collect from OWID.** The chart `annual-co2-emissions-per-country` gives total emissions in tonnes. Look at its two URLs in your browser first, then collect it with `get_owid`.
2. 🔵 **Prepare.** Lower-case the columns, recode `OWID_KOS`, keep three-letter codes, rename to `iso3`, `year` (integer) and `co2_tonnes`, keep 2020 to 2022, and check one row per country–year.
3. 🔵 **Collect from the World Bank.** Population (`SP.POP.TOTL`) for all countries, 2020 to 2022, with `wb_series`; drop aggregates and add income group (reuse `wb_table` and `countries`).
4. 🔵 **Join** with `how="outer"`, `validate="one_to_one"` and `indicator=True`. List the OWID-only codes and keep `both`.
5. 🔵 **Validate:** share of OWID rows matched, no duplicated keys, no negative totals, missing values per column.
6. 🔵 **Cross-source check.** Compute `co2_per_capita_wb = co2_tonnes / population`. Also collect OWID's per-person series for 2020 to 2022 (as in C1), join it on, and compute the **percentage difference** between the two per-person figures. List the five countries with the largest difference.
7. 🔵 **Answer the question:** each income group's share of total emissions in 2022, as a percentage.

Store the final table in `totals` (columns `iso3`, `year`, `co2_tonnes`, `population`, `income_group`, `co2_per_capita_wb`, ...).

In [36]:
# 1. Collect from OWID
tot_raw, tot_meta = get_owid("annual-co2-emissions-per-country")

# 2. Prepare
def prepare_owid(raw, value_name, years):
    raw = raw.copy()
    raw.columns = raw.columns.str.lower()
    col = [c for c in raw.columns if c not in ("entity", "code", "year")][0]
    t = raw.assign(code=raw["code"].replace(OWID_TO_WB))
    t = t[t["code"].fillna("").str.fullmatch(r"[A-Z]{3}")]
    t = t.rename(columns={"code": "iso3", col: value_name})[["iso3", "year", value_name]]
    t["year"] = t["year"].astype(int)
    t = t[t["year"].isin(years)]
    check(not t.duplicated(["iso3", "year"]).any(), "One row per country-year.")
    return t

tot = prepare_owid(tot_raw, "co2_tonnes", range(2020, 2023))

# 3. Collect from the World Bank
pop3 = wb_table(wb_series("SP.POP.TOTL", "2020:2022"), "population")
pop3 = pop3[pop3["iso3"].isin(countries["iso3"])].merge(countries, on="iso3", how="left")

# 4. Join
j = pd.merge(tot, pop3, on=["iso3", "year"], how="outer", validate="one_to_one", indicator=True)
print(j["_merge"].value_counts())
print("OWID only:", sorted(j.loc[j["_merge"] == "left_only", "iso3"].unique()))
totals = j[j["_merge"] == "both"].drop(columns="_merge")

# 5. Validate
print(f"Share of OWID rows matched: {(j['_merge'] == 'both').sum() / len(tot):.1%}")
check((totals["co2_tonnes"].dropna() >= 0).all(), "Totals should not be negative.")
print(totals.isna().sum())

# 6. Cross-source check
totals["co2_per_capita_wb"] = totals["co2_tonnes"] / totals["population"]
pc_raw, _ = get_owid("co-emissions-per-capita")
pc = prepare_owid(pc_raw, "co2_per_capita_owid", range(2020, 2023))
cross = totals.merge(pc, on=["iso3", "year"], how="inner", validate="one_to_one")
cross["pct_difference"] = 100 * (cross["co2_per_capita_wb"] - cross["co2_per_capita_owid"]) / cross["co2_per_capita_owid"]
print(cross["pct_difference"].describe().round(2))
print(cross.reindex(cross["pct_difference"].abs().sort_values(ascending=False).index)
      [["country", "year", "co2_per_capita_wb", "co2_per_capita_owid", "pct_difference"]].head())

# 7. Answer
y = totals[totals["year"] == 2022]
(100 * y.groupby("income_group")["co2_tonnes"].sum() / y["co2_tonnes"].sum()).round(1)

https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?v=1&csvType=full&useColumnShortNames=true
https://ourworldindata.org/grapher/annual-co2-emissions-per-country.metadata.json?v=1&csvType=full&useColumnShortNames=true
data: status 200, content type text/csv
meta: status 200, content type application/json
✅ Correct
https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=1000&date=2020%3A2022&page=1
_merge
both          615
right_only     36
left_only      33
Name: count, dtype: int64
OWID only: ['AIA', 'ATA', 'BES', 'COK', 'CXR', 'MSR', 'NIU', 'SHN', 'SPM', 'TWN', 'WLF']
Share of OWID rows matched: 94.9%
✅ Correct
iso3            0
year            0
co2_tonnes      0
population      0
country         0
income_group    0
dtype: int64
https://ourworldindata.org/grapher/co-emissions-per-capita.csv?v=1&csvType=full&useColumnShortNames=true
https://ourworldindata.org/grapher/co-emissions-per-capita.metadata.json?v=1&csvType=full&useColumnShortN

,co2_tonnes
income_group,
High income,38.6
Low income,0.6
Lower middle income,11.6
Upper middle income,49.2


In [37]:
# Test for C5
check({"iso3", "year", "co2_tonnes", "population", "income_group", "co2_per_capita_wb"}.issubset(totals.columns), "Name the columns as asked.")
check(totals["year"].between(2020, 2022).all() and totals["year"].dtype.kind == "i", "Years 2020-2022, as integers.")
check(not totals.duplicated(["iso3", "year"]).any(), "One row per country-year after the join.")
check(not totals["iso3"].isin(["WLD", "HIC", "LIC", "EUU"]).any() and not totals["iso3"].str.startswith("OWID").any(),
      "Drop aggregates and OWID group codes.")

✅ Correct
✅ Correct
✅ Correct
✅ Correct


✍️ **Interpret (three to four sentences).** For most countries, the two per-person figures should agree to within a few per cent. Why are they not identical? (Read OWID's metadata: whose population figures does OWID use?) For the countries with the largest differences, what might explain them? Which is the better choice for your analysis, and what would you write in your report?

*Your answer:*

## C6. Your turn: a reusable, validated join

A project repeats the same join for several indicators. Turn the checks into a function so they run every time.

🔵 Write `validated_join(left, right, keys=("iso3", "year"), min_match=0.9)` that:

1. checks neither table has duplicated keys (raise `ValueError` with a clear message if one does);
2. checks the key columns have the **same types** in both tables (raise `TypeError` if not);
3. runs an outer merge with `validate="one_to_one"` and `indicator=True`;
4. prints the counts of `both`, `left_only` and `right_only`, and the codes that are only on the left;
5. raises `ValueError` if the share of **left** rows that matched is below `min_match`;
6. returns the inner-joined table without the `_merge` column.

The tests try it on a correct join and on three broken ones.

In [38]:
def validated_join(left, right, keys=("iso3", "year"), min_match=0.9):
    keys = list(keys)
    for name, t in [("left", left), ("right", right)]:
        if t.duplicated(keys).any():
            raise ValueError(f"The {name} table has duplicated keys, e.g. {t[t.duplicated(keys)][keys].head(3).values.tolist()}")
    for k in keys:
        if left[k].dtype != right[k].dtype:
            raise TypeError(f"Key {k!r} is {left[k].dtype} on the left but {right[k].dtype} on the right")
    j = pd.merge(left, right, on=keys, how="outer", validate="one_to_one", indicator=True)
    print(j["_merge"].value_counts().to_dict())
    print("Left only:", sorted(j.loc[j["_merge"] == "left_only", keys[0]].unique()))
    rate = (j["_merge"] == "both").sum() / len(left)
    if rate < min_match:
        raise ValueError(f"Only {rate:.1%} of left rows matched (minimum {min_match:.0%})")
    return j[j["_merge"] == "both"].drop(columns="_merge")

In [39]:
# Tests for C6
ok = validated_join(co2, wb)
check(len(ok) == len(merged), "A correct join should give the same rows as C3.")

def raises(exc, f):
    try:
        f(); return False
    except exc:
        return True

check(raises(ValueError, lambda: validated_join(pd.concat([co2, co2.head(1)]), wb)), "Duplicated keys should raise ValueError.")
check(raises(TypeError, lambda: validated_join(co2.assign(year=co2["year"].astype(str)), wb)), "Text years against integer years should raise TypeError.")
check(raises(ValueError, lambda: validated_join(co2.assign(year=co2["year"] + 20), wb)), "A low match rate should raise ValueError.")

{'both': 1025, 'right_only': 60, 'left_only': 45}
Left only: ['AIA', 'BES', 'COK', 'MSR', 'NIU', 'SHN', 'SPM', 'TWN', 'WLF']
✅ Correct
✅ Correct
✅ Correct
{'right_only': 1085, 'left_only': 1070, 'both': 0}
Left only: ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM', 'CPV', 'CRI', 'CUB', 'CUW', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'FRO', 'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GRL', 'GTM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LIE', 'LKA', 'LSO'

### Taking this to your project

Your project follows the same stages with a health outcome from the **WHO Global Health Observatory API**. The pipeline is the same; the API details differ, so read its documentation the way you read OWID's in C1: how the URL is built, how rows are filtered, how results are paged, which dimensions (such as sex or age group) each indicator has, and how countries are distinguished from regions. Then reuse `validated_join` and the five checks.

---
## Before you leave

- [ ] Parts A and B run, and every test shows ✅.
- [ ] You can explain the three pieces of a request, the five professional steps, and how OWID's URLs and CSV response differ from the World Bank's JSON.
- [ ] C1 to C4 run, and you can explain each validation check and why the denominator is population.
- [ ] You have started C5; C6 is done before your project collection starts.
- [ ] **Edit → Clear all outputs**, download the notebook into your personal repository, and commit it with a clear message. Keep `data/raw/` out of Git; commit `data/provenance.jsonl`.